# Notebook 2 — Validate the Hardy–Littlewood / Wolf theory layer

**Prerequisite.** Notebook 1 passed: the C sieve produces the correct `gaps.csv` and `records.csv` at $N = 10^{8}$ (verified row-by-row against an independent numpy sieve).

**Goal.** The public `summary.md` claims

| $g$ | empirical | Wolf | $\mathrm{emp}/\mathrm{Wolf}$ |
|---:|---:|---:|---:|
| 2  | 440,312 | 380,676 | **1.157** |
| 4  | 440,257 | 329,830 | **1.335** |
| 6  | 768,752 | 371,755 | **2.068** |
| 12 | 538,382 | 187,386 | **2.873** |

Either the theory is off-scale at this $N$, or the implementation of $C_g$ / $\mathrm{Li}_2$ / the Wolf formula has a bug. This notebook rebuilds every ingredient from scratch and checks:

1. **Hardy–Littlewood constant $C_g$** — independent factorisation; cross-check vs known analytic values ($C_4 = C_2$, $C_6 = 2 C_2$, $C_{10} = (4/3) C_2$, $C_{30} = (8/3) C_2$).
2. **Integral $\mathrm{Li}_2(N) = \int_2^N dt / \ln^2 t$** — `scipy.integrate.quad` vs a hand-rolled Simpson rule on a log-grid; both must agree to many digits.
3. **H-L pair count** $\pi_g(N) \sim C_g \cdot \mathrm{Li}_2(N)$ — counts *all* pairs $(p, p{+}g)$.
4. **Wolf (1998) consecutive-gap formula** $N_g(N) \sim C_g \cdot \mathrm{Li}_2(N) \cdot \exp(-g C_g / \ln N)$ — counts only *consecutive* primes.
5. **Empirical pair count**, computed from the sieve **directly** (not from the histogram), and compared against the histogram from notebook 1. This is the strongest cross-check: the empirical `emp/Wolf` comes from two fully independent sources.
6. **Twin–cousin identity** — $C_2 = C_4$, so $\pi_2(N) \approx \pi_4(N)$ must hold to Poisson precision.

If all six cross-checks pass and the `emp/Wolf` ratios still drift to ~2.8 at $g=12$, the discrepancy is physics (pre-asymptotic regime: Wolf is justified when $\ln N \gg g$, we have $\ln N \approx 18.4$). If any cross-check fails, the failing piece is the bug.

In [1]:
import sys, os, time, math, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import integrate

print('python', sys.version.split()[0], '| numpy', np.__version__,
      '| pandas', pd.__version__)

# --- Configuration -----------------------------------------------------------
N_LIMIT  = 10**8
NB_DIR   = Path.cwd()
C_BINARY = NB_DIR / 'main-csv-static'
WORK_DIR = NB_DIR / 'run_N1e8'
# -----------------------------------------------------------------------------

# Twin-prime constant, OEIS A005597 (2 * C_2_twin = Hardy-Littlewood C_2).
C2_TWIN = 0.6601618158468695739278121100145557784326233602847334133194484233354
C2      = 2 * C2_TWIN     # Hardy-Littlewood C_2 for pairs (p, p+2)
print(f'C_2 (HL)  = {C2:.15f}')
print(f'C_2_twin  = {C2_TWIN:.15f}')

python 3.11.10 | numpy 2.0.2 | pandas 2.2.3
C_2 (HL)  = 1.320323631693739
C_2_twin  = 0.660161815846870


## 1. Rebuild primes (cache-friendly — reuse notebook 1's sieve)


In [2]:
def sieve_primes(n: int) -> np.ndarray:
    if n < 2:
        return np.array([], dtype=np.int64)
    is_prime = np.ones(n + 1, dtype=bool)
    is_prime[:2] = False
    is_prime[4::2] = False
    for i in range(3, int(math.isqrt(n)) + 1, 2):
        if is_prime[i]:
            is_prime[i*i::2*i] = False
    return np.flatnonzero(is_prime).astype(np.int64)

t0 = time.perf_counter()
primes = sieve_primes(N_LIMIT)
print(f'sieve {N_LIMIT:.0e} in {time.perf_counter()-t0:.2f}s,  pi(N) = {len(primes):,}')
gaps = np.diff(primes)
print(f'total gaps: {len(gaps):,}   max gap: {gaps.max()}')

sieve 1e+08 in 0.41s,  pi(N) = 5,761,455
total gaps: 5,761,454   max gap: 220


## 2. Hardy–Littlewood constant $C_g$ — two independent implementations

$$C_g = 2 C_2^{\text{twin}} \prod_{\substack{p \mid g/2 \\ p > 2}} \frac{p-1}{p-2}$$

Implementation A: trial division. Implementation B: sympy's symbolic factorisation. They must agree to machine precision.

Known closed-form values we can anchor against:

| $g$ | $C_g / C_2$ | reason |
|---:|---:|:---|
| 2  | 1     | no odd prime divides $k=1$ |
| 4  | 1     | $k=2$, no odd prime |
| 6  | 2     | $k=3$, factor $(3-1)/(3-2)=2$ |
| 8  | 1     | $k=4=2^2$ |
| 10 | 4/3   | $k=5$, factor $4/3$ |
| 12 | 2     | $k=6$, only 3 contributes |
| 30 | 8/3   | $k=15=3{\cdot}5$, factor $2 {\cdot} 4/3$ |

In [3]:
def C_g_trial(g: int) -> float:
    if g <= 0 or g % 2 != 0: return 0.0
    k = g // 2
    factor = 1.0
    d, m = 3, k
    while d * d <= m:
        if m % d == 0:
            while m % d == 0:
                m //= d
            factor *= (d - 1) / (d - 2)
        d += 2
    if m > 2:
        factor *= (m - 1) / (m - 2)
    return C2 * factor

try:
    from sympy import factorint
    def C_g_sympy(g: int) -> float:
        if g <= 0 or g % 2 != 0: return 0.0
        k = g // 2
        factor = 1.0
        for p, _ in factorint(k).items():
            if p > 2:
                factor *= (p - 1) / (p - 2)
        return C2 * factor
    HAVE_SYMPY = True
except ImportError:
    HAVE_SYMPY = False

print(f'{"g":>4} | {"C_g trial":>12} | {"C_g sympy":>12} | {"C_g/C_2":>10} | expected')
expected = {2:1, 4:1, 6:2, 8:1, 10:4/3, 12:2, 18:2, 24:2, 30:8/3, 42:3}
for g, ratio_expected in expected.items():
    a = C_g_trial(g)
    b = C_g_sympy(g) if HAVE_SYMPY else float('nan')
    ratio = a / C2
    ok = abs(ratio - ratio_expected) < 1e-12
    sympy_ok = (not HAVE_SYMPY) or abs(a - b) < 1e-12
    flag = 'OK' if (ok and sympy_ok) else 'FAIL'
    print(f'{g:>4} | {a:>12.8f} | {b:>12.8f} | {ratio:>10.6f} | {ratio_expected:.6f}  {flag}')

   g |    C_g trial |    C_g sympy |    C_g/C_2 | expected
   2 |   1.32032363 |   1.32032363 |   1.000000 | 1.000000  OK
   4 |   1.32032363 |   1.32032363 |   1.000000 | 1.000000  OK
   6 |   2.64064726 |   2.64064726 |   2.000000 | 2.000000  OK
   8 |   1.98048545 |   1.32032363 |   1.500000 | 1.000000  FAIL
  10 |   1.76043151 |   1.76043151 |   1.333333 | 1.333333  OK
  12 |   1.65040454 |   2.64064726 |   1.250000 | 2.000000  FAIL
  18 |   2.64064726 |   2.64064726 |   2.000000 | 2.000000  OK
  24 |   3.96097090 |   2.64064726 |   3.000000 | 2.000000  FAIL
  30 |   3.52086302 |   3.52086302 |   2.666667 | 2.666667  OK
  42 |   3.16877672 |   3.16877672 |   2.400000 | 3.000000  FAIL


## 3. Li₂ integral — two independent numerical schemes

In [4]:
def Li2_quad(N: float) -> float:
    val, _ = integrate.quad(lambda t: 1.0 / np.log(t)**2, 2.0, N, limit=200)
    return float(val)

def Li2_simpson(N: float, n_panels: int = 200_000) -> float:
    # Uniform grid on u = log(t): t = e^u, dt = e^u du, integrand = e^u / u^2.
    u0, u1 = math.log(2.0), math.log(N)
    if n_panels % 2: n_panels += 1
    u = np.linspace(u0, u1, n_panels + 1)
    f = np.exp(u) / (u * u)
    h = (u1 - u0) / n_panels
    return float(h / 3.0 * (f[0] + f[-1] + 4 * f[1:-1:2].sum() + 2 * f[2:-2:2].sum()))

A = Li2_quad(N_LIMIT)
B = Li2_simpson(N_LIMIT)
print(f'Li2(N)  scipy.quad  = {A:.10e}')
print(f'Li2(N)  Simpson     = {B:.10e}')
print(f'rel diff            = {abs(A-B)/A:.3e}   (should be << 1e-6)')

# Also: a sanity check at a tiny scale where we know pi(100)=25 and can compare
# Li2(100) against the classical offset logarithmic integral li(x) - li(2).
print(f'\nSmall-scale: Li2(100) via quad = {Li2_quad(100):.4f}')

Li2(N)  scipy.quad  = 3.3353019188e+05
Li2(N)  Simpson     = 3.3353019188e+05
rel diff            = 1.874e-13   (should be << 1e-6)

Small-scale: Li2(100) via quad = 10.2516


## 4. Empirical pair counts — recomputed directly from primes

Two independent empirical counts for every $g$:

- **consecutive** — number of $i$ with `primes[i+1] - primes[i] == g` (this is what `gaps.csv` reports; must agree with our histogram).
- **all pairs** — number of primes $p \le N - g$ such that $p + g$ is also prime (H-L counts these, even if another prime lies strictly between $p$ and $p+g$).

We compute **both** from the prime array and compare the consecutive count with the notebook-1 histogram as a triple-check.

In [5]:
# Fast membership test: prime set as a boolean array indexed by integer.
is_prime = np.zeros(N_LIMIT + 1, dtype=bool)
is_prime[primes] = True

G_LIST = [2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 24, 30]

# consecutive: np.diff counts; fastest via bincount
hist_all = np.bincount(gaps, minlength=max(G_LIST) + 1)

# all-pair: p prime AND p+g prime, with p+g <= N
rows = []
for g in G_LIST:
    consec = int(hist_all[g]) if g < len(hist_all) else 0
    # all pairs
    p_ok = primes[primes + g <= N_LIMIT]
    pair  = int(is_prime[p_ok + g].sum())
    rows.append({'g': g, 'emp_consec': consec, 'emp_pair': pair,
                 'C_g': C_g_trial(g)})
emp = pd.DataFrame(rows)
emp

,g,emp_consec,emp_pair,C_g
0,2,440312,440312,1.320324
1,4,440257,440258,1.320324
2,6,768752,879908,2.640647
3,8,334180,439908,1.980485
4,10,430016,586811,1.760432
5,12,538382,880196,1.650405
6,14,293201,528095,1.584388
7,16,215804,441055,1.540378
8,18,384738,880443,2.640647
9,20,202922,586267,1.485364


## 5. Theoretical predictions and the `emp/Wolf` ratio

In [6]:
Li2_N = Li2_quad(N_LIMIT)
lnN   = math.log(N_LIMIT)

emp['HL_pair'] = emp['C_g'] * Li2_N
emp['Wolf']    = emp['HL_pair'] * np.exp(-emp['g'] * emp['C_g'] / lnN)
emp['emp_consec / Wolf']   = emp['emp_consec'] / emp['Wolf']
emp['emp_pair   / HL_pair'] = emp['emp_pair']   / emp['HL_pair']

display(emp[['g', 'emp_consec', 'emp_pair', 'HL_pair', 'Wolf',
             'emp_consec / Wolf', 'emp_pair   / HL_pair']].round(4))

print(f'\nln(N) = {lnN:.4f}  (Wolf is justified when ln N >> g)')

,g,emp_consec,emp_pair,HL_pair,Wolf,emp_consec / Wolf,emp_pair / HL_pair
0,2,440312,440312,4.403678e+05,381556.1251,1.1540,0.9999
1,4,440257,440258,4.403678e+05,330598.8279,1.3317,0.9998
2,6,768752,879908,8.807356e+05,372651.4577,2.0629,0.9991
3,8,334180,439908,6.605517e+05,279488.5932,1.1957,0.6660
4,10,430016,586811,5.871571e+05,225791.0996,1.9045,0.9994
5,12,538382,880196,5.504597e+05,187843.9281,2.8661,1.5990
6,14,293201,528095,5.284414e+05,158502.8174,1.8498,0.9993
7,16,215804,441055,5.137624e+05,134801.8599,1.6009,0.8585
8,18,384738,880443,8.807356e+05,66714.0617,5.7670,0.9997
9,20,202922,586267,4.954138e+05,98758.7240,2.0547,1.1834



ln(N) = 18.4207  (Wolf is justified when ln N >> g)


## 6. Cross-check against the claimed values in `summary.md`

If our independently computed Wolf numbers match `summary.md`'s Wolf column, the original Python implementation is not buggy. The remaining gap between empirical and Wolf is then pre-asymptotic physics.

In [7]:
claimed = pd.DataFrame([
    {'g':2,  'emp_claim':440312, 'Wolf_claim':380676, 'HL_claim':439361},
    {'g':4,  'emp_claim':440257, 'Wolf_claim':329830, 'HL_claim':439361},
    {'g':6,  'emp_claim':768752, 'Wolf_claim':371755, 'HL_claim':878722},
    {'g':8,  'emp_claim':334180, 'Wolf_claim':278816, 'HL_claim':659042},
    {'g':10, 'emp_claim':430016, 'Wolf_claim':225245, 'HL_claim':585815},
    {'g':12, 'emp_claim':538382, 'Wolf_claim':187386, 'HL_claim':549201},
])

cmp = emp.merge(claimed, on='g')
cmp['emp_match']  = cmp['emp_consec'] == cmp['emp_claim']
cmp['Wolf_ratio'] = cmp['Wolf'] / cmp['Wolf_claim']
cmp['HL_ratio']   = cmp['HL_pair'] / cmp['HL_claim']
display(cmp[['g','emp_consec','emp_claim','emp_match',
             'Wolf','Wolf_claim','Wolf_ratio',
             'HL_pair','HL_claim','HL_ratio']].round(4))

,g,emp_consec,emp_claim,emp_match,Wolf,Wolf_claim,Wolf_ratio,HL_pair,HL_claim,HL_ratio
0,2,440312,440312,True,381556.1251,380676,1.0023,440367.7942,439361,1.0023
1,4,440257,440257,True,330598.8279,329830,1.0023,440367.7942,439361,1.0023
2,6,768752,768752,True,372651.4577,371755,1.0024,880735.5885,878722,1.0023
3,8,334180,334180,True,279488.5932,278816,1.0024,660551.6913,659042,1.0023
4,10,430016,430016,True,225791.0996,225245,1.0024,587157.0590,585815,1.0023
5,12,538382,538382,True,187843.9281,187386,1.0024,550459.7428,549201,1.0023


## 7. Twin vs cousin (Poisson-level identity)

$C_2 = C_4$, so $\pi_2(N)$ and $\pi_4(N)$ must match to within $\sim\sqrt{N}$ fluctuations. `summary.md` reports $z = 0.06\sigma$, which is consistent with this — we reproduce.

In [8]:
c2 = int(hist_all[2])
c4 = int(hist_all[4])
diff = abs(c2 - c4)
sigma = math.sqrt(c2 + c4)
print(f'twin (g=2)  : {c2:,}')
print(f'cousin(g=4) : {c4:,}')
print(f'|diff|      : {diff:,}  = {diff/sigma:.2f} sigma  ({100*diff/c2:.4f}%)')
print(f'C_4 / C_2   : {C_g_trial(4)/C_g_trial(2):.15f}  (must be 1.0)')

twin (g=2)  : 440,312
cousin(g=4) : 440,257
|diff|      : 55  = 0.06 sigma  (0.0125%)
C_4 / C_2   : 1.000000000000000  (must be 1.0)


## 8. Verdict

Tick after the run:

- [ ] Section 2: every row `OK` (C_g matches trial + sympy + closed form).
- [ ] Section 3: `quad` and Simpson agree to ~1e-8 relative.
- [ ] Section 4: `emp_consec` equals the histogram from notebook 1 for every $g$.
- [ ] Section 6: our `Wolf` equals the `Wolf_claim` from `summary.md` (ratio column $\approx 1.0000$). `emp_match` column all `True`.
- [ ] Section 7: `C_4 / C_2 = 1` exactly; twin/cousin difference $< 1\sigma$.

If all green: the Python theory layer is **correct**. The large `emp/Wolf` ratios at $g \in \{6, 10, 12\}$ are pre-asymptotic (Wolf requires $\ln N \gg g$; we have $\ln N \approx 18.4$), not a bug. We can then move to notebook 3 (sequence-level statistics: autocorrelation, running minimum $g/\ln p$, jumping-champion transitions) with confidence.

If Section 6's `Wolf_ratio` $\ne 1$, the original implementation is the bug — our numbers are the reference. Paste the table.